In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
DATA = Path("../data")
OUT = Path("../python/output")
OUT.mkdir(exist_ok=True)

In [3]:
orders = pd.read_csv(DATA/"orders.csv", parse_dates=["Order_Date"])
ship = pd.read_csv(DATA/"shipments.csv", parse_dates=["Order_Date","Ship_Date","Delivery_Date","Promised_Delivery_Date"])
products = pd.read_csv(DATA/"products.csv")
warehouses = pd.read_csv(DATA/"warehouses.csv")
inventory = pd.read_csv(DATA/"inventory_snapshot.csv", parse_dates=["Snapshot_Month"])

In [4]:
print("Shapes:", orders.shape, ship.shape, products.shape, warehouses.shape, inventory.shape)
print("\nMissing values:")
print(orders.isna().sum())
print(ship.isna().sum())

Shapes: (20000, 8) (20000, 11) (60, 5) (8, 4) (5760, 6)

Missing values:
Order_ID        0
Order_Date      0
Customer_ID     0
Product_ID      0
Warehouse_ID    0
Quantity        0
Unit_Price      0
Order_Value     0
dtype: int64
Order_ID                    0
Order_Date                  0
Warehouse_ID                0
Quantity                    0
Ship_Date                   0
Delivery_Date             354
Promised_Delivery_Date    354
Delivery_Days             354
Status                      0
Delay_Days                354
Product_ID                  0
dtype: int64


In [5]:
# 1. Monthly orders and revenue
monthly = (orders.assign(Month=orders["Order_Date"].dt.to_period("M").astype(str))
           .groupby("Month")
           .agg(Orders=("Order_ID","count"), Units=("Quantity","sum"), Revenue=("Order_Value","sum"))
           .reset_index())

In [6]:
print("\nMonthly performance:")
print(monthly)


Monthly performance:
      Month  Orders  Units     Revenue
0   2025-01    1726   5426  2309969.71
1   2025-02    1574   4917  2123207.66
2   2025-03    1690   5421  2324020.35
3   2025-04    1677   5240  2285992.08
4   2025-05    1698   5525  2403265.76
5   2025-06    1637   5312  2303472.00
6   2025-07    1742   5665  2429911.87
7   2025-08    1694   5580  2446681.59
8   2025-09    1636   5256  2291484.63
9   2025-10    1649   5963  2559932.04
10  2025-11    1559   5775  2508820.95
11  2025-12    1718   6260  2770298.33


In [7]:
plt.figure(figsize=(10,5))
plt.plot(monthly["Month"], monthly["Orders"], marker="o")
plt.xticks(rotation=45)
plt.title("Monthly Order Volume")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.tight_layout()
plt.savefig(OUT/"monthly_orders.png", dpi=150)
plt.close()

In [8]:
# 2. Delivery performance
valid = ship[ship["Status"].isin(["On Time","Delayed"])].copy()
on_time_pct = 100 * (valid["Status"].eq("On Time").mean())
avg_delivery = valid["Delivery_Days"].mean()
print(f"\nOn-time delivery: {on_time_pct:.2f}%")
print(f"Average delivery days: {avg_delivery:.2f}")


On-time delivery: 60.94%
Average delivery days: 4.88


In [9]:
warehouse_perf = (ship[ship["Status"].isin(["On Time","Delayed"])]
                  .groupby("Warehouse_ID")
                  .agg(Shipments=("Order_ID","count"),
                       Avg_Delivery_Days=("Delivery_Days","mean"),
                       On_Time_Pct=("Status", lambda x: 100*(x=="On Time").mean()),
                       Avg_Delay_Days=("Delay_Days","mean"))
                  .reset_index()
                  .sort_values("On_Time_Pct"))

In [10]:
print("\nWarehouse performance:")
print(warehouse_perf)


Warehouse performance:
  Warehouse_ID  Shipments  Avg_Delivery_Days  On_Time_Pct  Avg_Delay_Days
2          W03       2473           4.909826    58.956733        1.060251
7          W08       2439           5.042230    59.368594        1.100451
0          W01       2422           4.862923    60.280760        0.964492
1          W02       2450           4.891429    60.448980        0.981633
5          W06       2467           4.906364    60.924199        0.938387
3          W04       2486           4.832663    62.027353        0.938858
6          W07       2470           4.837247    62.307692        0.949393
4          W05       2439           4.746207    63.222632        0.881509


In [11]:
plt.figure(figsize=(9,5))
plt.bar(warehouse_perf["Warehouse_ID"], warehouse_perf["On_Time_Pct"])
plt.axhline(on_time_pct, linestyle="--", label="Network average")
plt.title("On-Time Delivery by Warehouse")
plt.ylabel("On-Time %")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"warehouse_on_time.png", dpi=150)
plt.close()

In [12]:
# 3. Category analysis
cat = (orders.merge(products[["Product_ID","Category"]], on="Product_ID")
       .groupby("Category")
       .agg(Orders=("Order_ID","count"), Units=("Quantity","sum"), Revenue=("Order_Value","sum"))
       .reset_index()
       .sort_values("Revenue", ascending=False))

In [13]:
print("\nCategory performance:")
print(cat)


Category performance:
      Category  Orders  Units     Revenue
5       Sports    3595  11974  6077431.62
2  Electronics    4328  14361  5640124.32
0       Beauty    4431  14671  5566944.80
1        Books    2048   6846  4306329.98
4      Kitchen    2978   9905  4187401.57
3         Home    2620   8583  2978824.68


In [14]:
plt.figure(figsize=(9,5))
plt.bar(cat["Category"], cat["Revenue"])
plt.xticks(rotation=30)
plt.title("Revenue by Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig(OUT/"category_revenue.png", dpi=150)
plt.close()

In [15]:
# 4. Inventory risk
inventory["Risk_Flag"] = np.where(
    (inventory["Stock_Level"] <= inventory["Reorder_Level"]) |
    (inventory["Days_of_Cover"] < 5),
    "At Risk", "Healthy"
)
risk_rate = 100 * inventory["Risk_Flag"].eq("At Risk").mean()
print(f"\nInventory records at risk: {risk_rate:.2f}%")


Inventory records at risk: 94.03%


In [16]:
risk_by_wh = (inventory.groupby("Warehouse_ID")
              .agg(Risk_Rate=("Risk_Flag", lambda x: 100*(x=="At Risk").mean()),
                   Avg_Days_Cover=("Days_of_Cover","mean"))
              .reset_index()
              .sort_values("Risk_Rate", ascending=False))
print(risk_by_wh)

  Warehouse_ID  Risk_Rate  Avg_Days_Cover
3          W04  95.555556        1.574653
6          W07  95.277778        1.595278
5          W06  94.583333        1.625875
7          W08  94.583333        1.675222
0          W01  93.611111        1.747583
4          W05  93.611111        1.734597
2          W03  92.500000        1.785056
1          W02  92.500000        1.786486


In [17]:
# 5. Insight extraction
print("\n--- Interview-ready observations ---")
print("Highest-delay warehouse:", warehouse_perf.iloc[0]["Warehouse_ID"])
print("Lowest-delay warehouse:", warehouse_perf.iloc[-1]["Warehouse_ID"])
print("Highest-revenue category:", cat.iloc[0]["Category"])
print("Highest inventory-risk warehouse:", risk_by_wh.iloc[0]["Warehouse_ID"])


--- Interview-ready observations ---
Highest-delay warehouse: W03
Lowest-delay warehouse: W05
Highest-revenue category: Sports
Highest inventory-risk warehouse: W04


In [18]:
# Save analytical tables
monthly.to_csv(OUT/"monthly_performance.csv", index=False)
warehouse_perf.to_csv(OUT/"warehouse_performance.csv", index=False)
cat.to_csv(OUT/"category_performance.csv", index=False)
risk_by_wh.to_csv(OUT/"inventory_risk_by_warehouse.csv", index=False)